## Conectar a Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## Preparar dataset de importación desde el 2023 al 2025

In [ ]:
from pathlib import Path
import pandas as pd
import csv
import time

# ============================================
# 1. Rutas
# ============================================
BASE_DIR = Path("/content/drive/MyDrive/Mod10/proyecto")
TXT_DIR = BASE_DIR / "data/raw/aduanas/Txt_Originales"
RAW_DIR = BASE_DIR / "data/raw/aduanas"
PROCESSED_DIR = BASE_DIR / "data/processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE_RAW = RAW_DIR / "importaciones_hs_filtrado_raw.csv"
OUTPUT_FILE_STD = PROCESSED_DIR / "importaciones_hs_filtrado_estandarizado.csv"

# ============================================
# 2. Encabezados oficiales
# ============================================
columnas = [
    "NUMENCRIPTADO","TIPO_DOCTO","ADU","FORM","FECVENCI","CODCOMUN","NUM_UNICO_IMPORTADOR",
    "CODPAISCON","DESDIRALM","CODCOMRS","ADUCTROL","NUMPLAZO","INDPARCIAL","NUMHOJINS",
    "TOTINSUM","CODALMA","NUM_RS","FEC_RS","ADUA_RS","NUMHOJANE","NUM_SEC","PA_ORIG","PA_ADQ",
    "VIA_TRAN","TRANSB","PTO_EMB","PTO_DESEM","TPO_CARGA","ALMACEN","FEC_ALMAC","FECRETIRO",
    "NU_REGR","ANO_REG","CODVISBUEN","NUMREGLA","NUMANORES","CODULTVB","PAGO_GRAV","FECTRA",
    "FECACEP","GNOM_CIA_T","CODPAISCIA","NUMRUTCIA","DIGVERCIA","NUM_MANIF","NUM_MANIF1",
    "NUM_MANIF2","FEC_MANIF","NUM_CONOC","FEC_CONOC","NOMEMISOR","NUMRUTEMI","DIGVEREMI",
    "GREG_IMP","REG_IMP","BCO_COM","CODORDIV","FORM_PAGO","NUMDIAS","VALEXFAB","MONEDA",
    "MONGASFOB","CL_COMPRA","TOT_ITEMS","FOB","TOT_HOJAS","COD_FLE","FLETE","TOT_BULTOS",
    "COD_SEG","SEGURO","TOT_PESO","CIF","NUM_AUT","FEC_AUT","GBCOCEN","ID_BULTOS","TPO_BUL1",
    "CANT_BUL1","TPO_BUL2","CANT_BUL2","TPO_BUL3","CANT_BUL3","TPO_BUL4","CANT_BUL4","TPO_BUL5",
    "CANT_BUL5","TPO_BUL6","CANT_BUL6","TPO_BUL7","CANT_BUL7","TPO_BUL8","CANT_BUL8","CTA_OTRO",
    "MON_OTRO","CTA_OTR1","MON_OTR1","CTA_OTR2","MON_OTR2","CTA_OTR3","MON_OTR3","CTA_OTR4",
    "MON_OTR4","CTA_OTR5","MON_OTR5","CTA_OTR6","MON_OTR6","CTA_OTR7","MON_OTR7","MON_178",
    "MON_191","FEC_501","VAL_601","FEC_502","VAL_602","FEC_503","VAL_603","FEC_504","VAL_604",
    "FEC_505","VAL_605","FEC_506","VAL_606","FEC_507","VAL_607","TASA","NCUOTAS","ADU_DI",
    "NUM_DI","FEC_DI","MON_699","MON_199","NUMITEM","DNOMBRE","DMARCA","DVARIEDAD","DOTRO1",
    "DOTRO2","ATR-5","ATR-6","SAJU-ITEM","AJU-ITEM","CANT-MERC","MERMAS","MEDIDA","PRE-UNIT",
    "ARANC-ALA","NUMCOR","NUMACU","CODOBS1","DESOBS1","CODOBS2","DESOBS2","CODOBS3","DESOBS3",
    "CODOBS4","DESOBS4","ARANC-NAC","CIF-ITEM","ADVAL-ALA","ADVAL","VALAD","OTRO1","CTA1",
    "SIGVAL1","VAL1","OTRO2","CTA2","SIGVAL2","VAL2","OTRO3","CTA3","SIGVAL3","VAL3","OTRO4",
    "CTA4","SIGVAL4","VAL4"
]

# ============================================
# 3. HS objetivo
# ============================================
HS_CODES = {'84181019', '84501111', '84501119', '85166030', '85163230', '85163100', '85163290', '85285290', '84509000', '84186910', '85285910', '84186990', '85167100', '85167920', '85161035', '84186110', '84186940', '85169000', '84182120', '85284200', '85287100', '85164000', '84501132', '85162100', '85165000', '85166019', '85167910', '85161050', '85286200', '84186930', '85287300', '84189100', '85161039', '84181013', '84186980', '85167930', '84501900', '85167990', '84186970', '85287220', '84182130', '84186960', '84501139', '84183000', '84186959', '84501200', '85162900', '84501122', '84186120', '85168000', '85287290', '85161034', '84181011', '84189900', '85163300', '84185000', '85285920', '84501190', '84501129', '85161049', '84186190', '84182110', '85286900', '84501131', '84502000', '85166012', '84182190', '85284900', '85161041', '85285230', '84501112', '85167200', '84186951', '84182900', '84181012', '84181090'}

# ============================================
# 4. Funciones auxiliares
# ============================================
def normalizar_hs_serie(serie):
    return (
        serie.fillna("")
        .astype(str)
        .str.replace(r"\D", "", regex=True)
        .str[:8]
        .str.strip()
    )


def convertir_numerico(serie):
    if serie is None:
        return None
    return pd.to_numeric(
        serie.fillna("")
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip(),
        errors="coerce"
    )


def convertir_fecha(serie):
    if serie is None:
        return pd.NaT

    s = (
        serie.fillna("")
        .astype(str)
        .str.strip()
        .str.replace("/", "-", regex=False)
    )

    fecha = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")

    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%d-%m-%Y %H:%M:%S", errors="coerce")

    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%Y-%m-%d", errors="coerce")

    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%Y-%m-%d %H:%M:%S", errors="coerce")

    mask = fecha.isna()
    if mask.any():
        fecha.loc[mask] = pd.to_datetime(s[mask], format="%d%m%Y", errors="coerce")

    return fecha

# ============================================
# 5. Archivos y configuración
# ============================================
archivos_txt = sorted(TXT_DIR.glob("*.txt"))

if not archivos_txt:
    raise FileNotFoundError(f"No se encontraron archivos TXT en {TXT_DIR}")

for archivo_salida in [OUTPUT_FILE_RAW, OUTPUT_FILE_STD]:
    if archivo_salida.exists():
        archivo_salida.unlink()

chunk_size = 100_000
separador_entrada = ";"
encoding_entrada = "latin1"

inicio = time.time()
total_filtrado = 0
archivos_ok = 0
archivos_error = []
primer_escritura_raw = True
primer_escritura_std = True

print("=" * 70)
print("INICIO DEL PROCESO OPTIMIZADO")
print("Se filtra por HS mientras se leen los TXT para evitar un consolidado gigante.")
print(f"Archivos encontrados: {len(archivos_txt)}")
print(f"Columnas esperadas: {len(columnas)}")
print(f"Salida raw filtrada: {OUTPUT_FILE_RAW}")
print(f"Salida estandarizada: {OUTPUT_FILE_STD}")
print("=" * 70)

# ============================================
# 6. Lectura por bloques + filtro temprano
# ============================================
for archivo in archivos_txt:
    print(f"\nProcesando: {archivo.name}")

    try:
        lector = pd.read_csv(
            archivo,
            sep=separador_entrada,
            encoding=encoding_entrada,
            header=None,
            dtype=str,
            chunksize=chunk_size,
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip"
        )

        for i, chunk in enumerate(lector, start=1):
            # Si el archivo repite encabezado dentro del contenido, se elimina.
            if len(chunk) > 0:
                fila0 = [str(x).strip() if pd.notna(x) else "" for x in chunk.iloc[0].tolist()]
                if fila0[:len(columnas)] == columnas:
                    chunk = chunk.iloc[1:].copy()

            if chunk.empty:
                continue

            # Ajustar ancho esperado del archivo.
            if chunk.shape[1] < len(columnas):
                chunk = chunk.reindex(columns=range(len(columnas)))
            elif chunk.shape[1] > len(columnas):
                chunk = chunk.iloc[:, :len(columnas)].copy()

            chunk.columns = columnas
            chunk["archivo_origen"] = archivo.name

            # Filtro temprano: primero se normaliza HS y luego se descarta lo no necesario.
            chunk["HS_CODE"] = normalizar_hs_serie(chunk["ARANC-NAC"])
            filtrado = chunk[chunk["HS_CODE"].isin(HS_CODES)].copy()

            if filtrado.empty:
                print(f"  Bloque {i}: 0 filas filtradas | Acumulado: {total_filtrado:,}")
                continue

            # Guardado raw filtrado.
            filtrado.to_csv(
                OUTPUT_FILE_RAW,
                sep=";",
                mode="w" if primer_escritura_raw else "a",
                index=False,
                header=primer_escritura_raw,
                encoding="utf-8-sig"
            )
            primer_escritura_raw = False

            # Dataset estandarizado final.
            df_std = pd.DataFrame()
            df_std["archivo_origen"] = filtrado["archivo_origen"]
            df_std["documento_id"] = filtrado["NUMENCRIPTADO"]
            df_std["num_item"] = filtrado["NUMITEM"]
            df_std["fecha"] = convertir_fecha(filtrado["FECTRA"])
            df_std["codigo_arancel"] = filtrado["ARANC-NAC"]
            df_std["hs_code"] = filtrado["HS_CODE"]
            df_std["hs4"] = filtrado["HS_CODE"].astype(str).str[:4]
            df_std["descripcion_producto"] = filtrado["DNOMBRE"]
            df_std["marca"] = filtrado["DMARCA"]
            df_std["unidad_medida"] = filtrado["MEDIDA"]
            df_std["cantidad"] = convertir_numerico(filtrado["CANT-MERC"])
            df_std["peso"] = convertir_numerico(filtrado["TOT_PESO"])
            df_std["valor_cif"] = convertir_numerico(filtrado["CIF-ITEM"])

            desc = df_std["descripcion_producto"].fillna("").astype(str).str.upper().str.strip().str[:40]
            marca = df_std["marca"].fillna("").astype(str).str.upper().str.strip()
            hs = df_std["codigo_arancel"].fillna("").astype(str).str.strip()
            df_std["sku_referencia"] = hs + "_" + marca + "_" + desc

            df_std.to_csv(
                OUTPUT_FILE_STD,
                sep=";",
                mode="w" if primer_escritura_std else "a",
                index=False,
                header=primer_escritura_std,
                encoding="utf-8-sig"
            )
            primer_escritura_std = False

            total_filtrado += len(filtrado)
            print(f"  Bloque {i}: {len(filtrado):,} filas filtradas | Acumulado: {total_filtrado:,}")

        archivos_ok += 1

    except Exception as e:
        print(f"  Error en {archivo.name}: {e}")
        archivos_error.append((archivo.name, str(e)))

fin = time.time()

print("\n" + "=" * 70)
print("PROCESO TERMINADO")
print("=" * 70)
print(f"Archivos procesados correctamente: {archivos_ok}")
print(f"Total registros filtrados: {total_filtrado:,}")
print(f"Archivo raw filtrado: {OUTPUT_FILE_RAW}")
print(f"Archivo estandarizado: {OUTPUT_FILE_STD}")
print(f"Tiempo total: {round(fin - inicio, 2)} segundos")

if archivos_error:
    print("\nArchivos con error:")
    for nombre, error in archivos_error:
        print(f"- {nombre}: {error}")



INICIO DEL PROCESO OPTIMIZADO
Se filtra por HS mientras se leen los TXT para evitar un consolidado gigante.
Archivos encontrados: 36
Columnas esperadas: 178
Salida raw filtrada: /content/drive/MyDrive/Mod10/proyecto/data/raw/aduanas/importaciones_hs_filtrado_raw.csv
Salida estandarizada: /content/drive/MyDrive/Mod10/proyecto/data/processed/importaciones_hs_filtrado_estandarizado.csv

Procesando: Importaciones abril 2023.txt
  Bloque 1: 1,076 filas filtradas | Acumulado: 1,076
  Bloque 2: 1,447 filas filtradas | Acumulado: 2,523
  Bloque 3: 855 filas filtradas | Acumulado: 3,378
  Bloque 4: 88 filas filtradas | Acumulado: 3,466

Procesando: Importaciones abril 2024.txt
  Bloque 1: 1,409 filas filtradas | Acumulado: 4,875
  Bloque 2: 1,466 filas filtradas | Acumulado: 6,341
  Bloque 3: 617 filas filtradas | Acumulado: 6,958
  Bloque 4: 412 filas filtradas | Acumulado: 7,370

Procesando: Importaciones abril 2025.txt
  Bloque 1: 1,323 filas filtradas | Acumulado: 8,693
  Bloque 2: 2,004 fi

## Validación rápida de salidas


In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Mod10/proyecto")
RAW_DIR = BASE_DIR / "data/raw/aduanas"
PROCESSED_DIR = BASE_DIR / "data/processed"

OUTPUT_FILE_RAW = RAW_DIR / "importaciones_2023_a_2025_filtrado_raw.csv"
OUTPUT_FILE_STD = PROCESSED_DIR / "importaciones_2023_a_2025_filtrado_estandarizado.csv"

for archivo in [OUTPUT_FILE_RAW, OUTPUT_FILE_STD]:
    if not archivo.exists():
        print(f"No existe: {archivo}")
        continue

    print("=" * 70)
    print(f"Archivo: {archivo.name}")
    muestra = pd.read_csv(archivo, sep=";", nrows=5, encoding="utf-8-sig")
    print(f"Columnas: {len(muestra.columns)}")
    print(f"Primeras columnas: {muestra.columns.tolist()[:10]}")
    display(muestra.head())



Archivo: importaciones_2023_a_2025_filtrado_raw.csv
Columnas: 180
Primeras columnas: ['NUMENCRIPTADO', 'TIPO_DOCTO', 'ADU', 'FORM', 'FECVENCI', 'CODCOMUN', 'NUM_UNICO_IMPORTADOR', 'CODPAISCON', 'DESDIRALM', 'CODCOMRS']


,NUMENCRIPTADO,TIPO_DOCTO,ADU,FORM,FECVENCI,CODCOMUN,NUM_UNICO_IMPORTADOR,CODPAISCON,DESDIRALM,CODCOMRS,...,OTRO3,CTA3,SIGVAL3,VAL3,OTRO4,CTA4,SIGVAL4,VAL4,archivo_origen,HS_CODE
0,20217189,101,48,15,18042023,5101,6876,336,NaN,13124,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones abril 2023.txt,85285910
1,20180907,101,39,15,12052023,13106,13578,336,NaN,5601,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones abril 2023.txt,85167990
2,20143304,101,48,15,25042023,13123,8682,225,NaN,13124,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones abril 2023.txt,84189900
3,20147669,101,48,15,2052023,13402,12153,333,NaN,13124,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones abril 2023.txt,85285910
4,20174245,101,48,15,29042023,13107,3465,225,NaN,13101,...,0,NaN,NaN,0,0,NaN,NaN,0,Importaciones abril 2023.txt,85285290


Archivo: importaciones_2023_a_2025_filtrado_estandarizado.csv
Columnas: 14
Primeras columnas: ['archivo_origen', 'documento_id', 'num_item', 'fecha', 'codigo_arancel', 'hs_code', 'hs4', 'descripcion_producto', 'marca', 'unidad_medida']


,archivo_origen,documento_id,num_item,fecha,codigo_arancel,hs_code,hs4,descripcion_producto,marca,unidad_medida,cantidad,peso,valor_cif,sku_referencia
0,Importaciones abril 2023.txt,20217189,1,2023-04-03,85285910,85285910,8528,SIN-CODIGO ~ DISPLAY~ SHENZHEN WANGSBROTH...,"~ W190-XQRL-RT-B~ EN COLOR, DE",10,13.0000,95.50,4973.00,"85285910_~ W190-XQRL-RT-B~ EN COLOR, DE_SIN-CO..."
1,Importaciones abril 2023.txt,20180907,4,2023-04-27,85167990,85167990,8516,SIN-CODIGO ~ CALENTADOR~ HUALI IMPORT & E...,T-F~ DE CERA~ ARTICULOS PARA U,10,564.0000,3591.84,402.00,85167990_T-F~ DE CERA~ ARTICULOS PARA U_SIN-CO...
2,Importaciones abril 2023.txt,20143304,3,2023-04-10,84189900,84189900,8418,SIN-CODIGO ~ DIFERENCIAL~ MIDWEST REFRIGE...,ON SYSTEMS-F~ DE PRESION~ PART,6,4.6923,30.20,675.24,84189900_ON SYSTEMS-F~ DE PRESION~ PART_SIN-CO...
3,Importaciones abril 2023.txt,20147669,1,2023-04-17,85285910,85285910,8528,SIN-CODIGO ~ PANTALLA TACTIL~ LEEPACK-F~ ...,P4501TAA~ DISPLAY EN COLOR,10,1.0000,11.33,1752.97,85285910_P4501TAA~ DISPLAY EN COLOR_SIN-CODIGO...
4,Importaciones abril 2023.txt,20174245,12,2023-04-14,85285290,85285290,8528,55337999 ~ MONITOR LED~ MSI~ MAG274QRF-...,7 PULG. WQHD 2560 X 1440~ EN C,10,11.0000,473.60,2216.48,85285290_7 PULG. WQHD 2560 X 1440~ EN C_553379...
